# Destiny Explained Studio — bibliothèque v1.0

Notebook additionnel, T4/Colab/Android non validés dans le dépôt. Choisir GPU T4 si disponible. Commencer avec une seule frame Draft, LOD0. Les prototypes restent partiels. Aucun épisode complet ni narration modifiée.


In [ ]:
# @title 1. Installer l’extension et Blender officiel (à exécuter une fois)
import hashlib, pathlib, subprocess, urllib.request
REPO = pathlib.Path('/content/Destiny-Explained-Studio')
BRANCH = 'feat/library-crota-lot-b-1.0'
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--single-branch','--branch',BRANCH,'https://github.com/Alexanst54/Destiny-Explained-Studio.git',str(REPO)],check=True)
else:
    actual = subprocess.check_output(['git','-C',str(REPO),'branch','--show-current'],text=True).strip()
    assert actual == BRANCH, 'Le dossier existe sur une autre branche ; utilisez un runtime neuf.'
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','libxi6','libxrender1','libxkbcommon0','libsm6','libice6','libgl1','libegl1','ffmpeg'],check=True)
archive = pathlib.Path('/content/blender-4.3.2-linux-x64.tar.xz')
base = 'https://download.blender.org/release/Blender4.3/'
BLENDER = pathlib.Path('/content/blender-4.3.2-linux-x64/blender')
if not BLENDER.exists():
    urllib.request.urlretrieve(base+archive.name,str(archive))
    checksums = urllib.request.urlopen(base+'blender-4.3.2.sha256').read().decode()
    checksum = hashlib.file_digest(archive.open('rb'),'sha256').hexdigest()
    assert any(checksum in line and archive.name in line for line in checksums.splitlines()), 'SHA-256 incorrect'
    subprocess.run(['tar','-xf',str(archive),'-C','/content'],check=True)
subprocess.run([str(BLENDER),'--version'],check=True)
try: subprocess.run(['nvidia-smi'],check=False)
except FileNotFoundError: print('Aucun pilote NVIDIA détecté ; fallback CPU disponible.')


In [ ]:
# @title 2. Sélection et rendu court
selection = 'boss-atheon' # @param {type:'string'}
# Choisir un ID exact de l’Atlas. Les boss sans modèle dédié sont refusés.
allow_partial = False # @param {type:'boolean'}
family_study = False # @param {type:'boolean'}
family = 'hive' # @param ['vex','hive','taken','cabal','eliksni','pyramid','bray','nine','neomuna','pale-heart']
view = 'environment' # @param ['front','three-quarter','silhouette','environment','reveal','close-up']
animation = 'idle' # @param ['idle','walk','move','attack','rotate','combat','temporal','reveal']
quality = 'draft' # @param ['draft','balanced','cinematic']
lod = 0 # @param [0,1,2] {type:'raw'}
frames = 1 # @param {type:'integer'}
width = 480 # @param {type:'integer'}
height = 360 # @param {type:'integer'}
output_name = 'library-preview' # @param {type:'string'}
resume = False # @param {type:'boolean'}
use_drive = False # @param {type:'boolean'}
import json, sys
sys.path.insert(0,str(REPO/'remotion/blender'))
from universal.request import request
r=request(selection,quality=quality,lod=lod,frames=frames,width=width,height=height,view=view,animation=animation,family_study=family_study,family=family if family_study else None)
assert allow_partial or not r['binding']['status'].startswith('partial'), 'Prototype/étude partiel : activer allow_partial pour l’examiner.'
assert 1 <= frames <= 24, 'Validation courte : maximum 24 frames.'
assert pathlib.Path(output_name).name == output_name and output_name not in ['.','..',''], 'Nom de dossier simple requis.'
base=pathlib.Path('/content/library-renders')
if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    base=pathlib.Path('/content/drive/MyDrive/Destiny-Explained-Studio/library-renders')
OUT=base/output_name
cmd=[sys.executable,str(REPO/'remotion/blender/render_universal.py'),selection,str(OUT),'--quality',quality,'--lod',str(lod),'--view',view,'--animation',animation,'--frames',str(frames),'--fps','24','--width',str(width),'--height',str(height),'--blender',str(BLENDER),'--timeout','7200']
if family_study:cmd+=['--family-study','--family',family]
if resume:cmd+=['--resume']
if frames>1:cmd+=['--video']
subprocess.run(cmd,check=True,cwd=REPO)


In [ ]:
# @title 3. Inspecter et télécharger
import json,shutil
from IPython.display import display,Image,Video
from google.colab import files
display(Image(filename=str(OUT/'frame-0000.png')))
if (OUT/'sequence.mp4').exists(): display(Video(str(OUT/'sequence.mp4'),embed=True))
report=json.loads((OUT/'progress.json').read_text())
print(json.dumps({k:report.get(k) for k in ['status','configuration','cacheHits','renderedThisRun','lastRunWallSeconds','peakProcessRssKiB','gpuMemoryAndUtilization']},indent=2))
print('Vérifier selected/backend/devices : les mesures CPU ne prouvent aucun gain GPU.')
archive=shutil.make_archive('/content/'+output_name,'zip',OUT)
files.download(archive)


## Reprise

Conserver exactement le dossier, les sources et les options, puis cocher `resume`. Les PNG achevés sont décodés et contrôlés avant réemploi. Pour survivre à la perte de `/content`, choisir Drive avant le premier rendu. Toute modification du code, de Blender ou des paramètres exige un autre dossier. La scène est reconstruite après redémarrage ; cache JSON et images seulement. Aucun modèle générique ne remplace un boss manquant.
